# EdU segmentation and cell classification

Classify existing cell labels using the EdU signal in the original image stacks,
then summarize intensities and export images with label masks.

Run this notebook from `Code/` after preparing these arrays in the same
kernel (or loading them from your segmentation results):
- `unprocessed_img`: raw images with axes `(Z, Y, X, C)`; EdU is channel 0.
- `labelArray`: matching integer cell-label volumes with axes `(Z, Y, X)`;
  label 0 is background.
- `mergedArray`: matching images with segmentation labels already appended,
  with axes `(Z, Y, X, C)`, used only by the optional segmentation export.

All arrays must follow the sorted input-file order. This notebook does not run
StarDist or reconstruct these arrays. Raw image data are not included in this
repository.

In [ ]:
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from csbdeep.io import save_tiff_imagej_compatible
from skimage import filters
from skimage.measure import regionprops
from skimage.morphology import ball, dilation

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams["image.interpolation"] = "none"

## Configure inputs and classification

Check the file pattern and genotype before running. The existing genotype value
is retained below, but the input directory is named `Experimental`.

A cell is EdU-positive when at least 10% of its dilated voxels exceed 1.2 times
the image-wide Otsu threshold. Intensities are measured on the raw EdU channel.

In [ ]:
input_pattern = "../Images/Pnut_labeling_Edu_replicate3/Experimental/*"
results_directory = Path("../Images/Pnut_labeling_Edu_replicate3/Results")
genotype = "Control"  # Set to the genotype represented by the input images.
edu_channel = 0
otsu_multiplier = 1.2
fraction_thresh = 0.1
dilation_radius = 1  # Radius in voxels; no physical-spacing correction is applied.

files = sorted(glob(input_pattern))
if not files:
    raise FileNotFoundError(f"No input files matched {input_pattern!r}.")

required_inputs = ("unprocessed_img", "labelArray")
missing_inputs = [name for name in required_inputs if name not in globals()]
if missing_inputs:
    raise RuntimeError(
        "Load the upstream segmentation arrays first: " + ", ".join(missing_inputs)
    )
if not (len(files) == len(unprocessed_img) == len(labelArray)):
    raise ValueError("Files, raw images, and label volumes must have equal lengths.")

for file_name, raw_img, labels in zip(files, unprocessed_img, labelArray):
    if raw_img.ndim != 4 or raw_img.shape[-1] <= edu_channel:
        raise ValueError(f"{file_name}: expected a ZYXC image with an EdU channel.")
    if labels.ndim != 3 or labels.shape != raw_img.shape[:3]:
        raise ValueError(f"{file_name}: label dimensions must match the raw image.")

## Classify cells and measure EdU intensity

Dilation is applied directly to the integer label image, preserving the original
method: where labels compete, grayscale dilation favors the larger label ID.
Areas and intensities below refer to these dilated regions, not the original
cell boundaries. Positive and negative masks retain the cell label IDs.

In [ ]:
img_data = []
positive_masks = []
negative_masks = []
dilated_labels_all = []

for i, (raw_img, labels) in enumerate(zip(unprocessed_img, labelArray)):
    edu = raw_img[..., edu_channel]

    edu_threshold = filters.threshold_otsu(edu) * otsu_multiplier
    edu_mask = edu > edu_threshold

    labels_dil = dilation(labels, ball(dilation_radius))
    dilated_labels_all.append(labels_dil)

    pos_mask = np.zeros_like(labels_dil, dtype=np.uint16)
    neg_mask = np.zeros_like(labels_dil, dtype=np.uint16)

    for prop in regionprops(labels_dil, intensity_image=edu):
        label_id = prop.label
        prop_mask = edu_mask[labels_dil == label_id]
        positive = np.mean(prop_mask) >= fraction_thresh

        # Measure raw intensity within the dilated cell region.
        img_data.append({
            'Genotype': genotype,
            'Sample': str(i),
            'label': label_id,
            'area': prop.area,  # Voxel count, not physical volume.
            'EdU_mean': prop.mean_intensity,
            'EdU_total': prop.mean_intensity * prop.area,
            'EdU_positive': positive
        })

        # Build masks
        if positive:
            pos_mask[labels_dil == label_id] = label_id
        else:
            neg_mask[labels_dil == label_id] = label_id

    positive_masks.append(pos_mask)
    negative_masks.append(neg_mask)

# Explicit columns keep downstream summaries valid when no cells are detected.
img_data = pd.DataFrame(img_data, columns=[
    "Genotype", "Sample", "label", "area", "EdU_mean", "EdU_total", "EdU_positive"
])
img_data["EdU_positive"] = img_data["EdU_positive"].astype(bool)

## Summarize cell counts

Sample IDs are zero-based positions in the sorted input-file list.

In [ ]:
total_cells = len(img_data)
num_pos = img_data['EdU_positive'].sum()
num_neg = total_cells - num_pos

percent_pos = 100 * num_pos / total_cells if total_cells else 0.0
percent_neg = 100 * num_neg / total_cells if total_cells else 0.0

print(f"Total cells: {total_cells}")
print(f"EdU-positive: {num_pos} ({percent_pos:.1f}%)")
print(f"EdU-negative: {num_neg} ({percent_neg:.1f}%)")

summary = (
    img_data
    .groupby("Sample")["EdU_positive"]
    .agg(
        total_cells="count",
        num_pos="sum"
    )
)
summary["num_neg"] = summary["total_cells"] - summary["num_pos"]
summary["percent_pos"] = summary["num_pos"] / summary["total_cells"] * 100

print(summary)

## Export raw images with cell and EdU-positive masks

Append the original (undilated) cell labels and the dilated EdU-positive mask to
the raw channels. Three raw channels produce five output channels. The historical
`_5ch_wmasks.tif` suffix is retained. Existing files are overwritten.

In [ ]:
edu_directory = results_directory / "Edu"
edu_directory.mkdir(parents=True, exist_ok=True)

for file_path, raw_img, labels, pos_mask in zip(
    files, unprocessed_img, labelArray, positive_masks
):
    # Preserve the original export's uint16 label channels.
    cell_channel = labels[..., np.newaxis].astype(np.uint16)
    positive_channel = pos_mask[..., np.newaxis].astype(np.uint16)
    final_img = np.concatenate((raw_img, cell_channel, positive_channel), axis=-1)

    output_path = edu_directory / f"{Path(file_path).stem}_5ch_wmasks.tif"
    save_tiff_imagej_compatible(output_path, final_img, axes="ZYXC")

## Inspect intensity distributions

In [ ]:
# Loop over each sample separately
for sample_id, df_sub in img_data.groupby("Sample"):
    pos_mean = df_sub.loc[df_sub['EdU_positive'], 'EdU_mean']
    neg_mean = df_sub.loc[~df_sub['EdU_positive'], 'EdU_mean']

    plt.figure(figsize=(6, 4))
    plt.hist(pos_mean, bins=30, alpha=0.5, label='Positive')
    plt.hist(neg_mean, bins=30, alpha=0.5, label='Negative')
    plt.xlabel('Mean EdU intensity per cell')
    plt.ylabel('Count')
    plt.title(f'Sample {sample_id}')
    plt.legend()
    plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
pos_mean = img_data.loc[img_data['EdU_positive'], 'EdU_mean']
neg_mean = img_data.loc[~img_data['EdU_positive'], 'EdU_mean']


plt.hist(pos_mean, bins=30, alpha=0.5, label='Positive')
plt.hist(neg_mean, bins=30, alpha=0.5, label='Negative')
plt.xlabel('Mean EdU intensity per cell')
plt.ylabel('Count')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(
    data=img_data,
    x='Sample',
    y='EdU_mean',
    hue='EdU_positive',
    legend=False,
    palette={True: 'red', False: 'blue'},
    alpha=0.7
)
plt.xlabel('Sample')
plt.ylabel('EdU mean intensity')
plt.title('EdU-positive vs negative cells')
plt.show()